In [1]:
import os

import numpy as np
import torch
from dotenv import load_dotenv
from hydra import compose
from hydra.core.global_hydra import GlobalHydra
from hydra.initialize import initialize_config_dir
from hydra.utils import instantiate
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

from sklearn.utils import resample
from datasets import load_dataset

from flower.evaluation.metrics import prepare_data, evaluate_embedding_classifier, evaluate_embedding_regressor
from flower.evaluation.benchmarks import resiualize_categorical

load_dotenv()
DATA_ROOT = os.getenv("DATA_ROOT")
RANDOM_STATE = 42 

/home/phys2526/WhatWeDontCatalog/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset(
    "parquet",
    data_files={
        "train": f"{DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior/embeddings/7518770_0/train/*.parquet",
        "test": f"{DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior/embeddings/7518770_0/test/*.parquet"
    }
)

In [26]:
clf_models = {
    'log_regression': LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000
    ),
    '2-mlp': MLPClassifier(
        hidden_layer_sizes=(64, 32), 
        max_iter=1000, 
        random_state=RANDOM_STATE
    )
}

reg_models = {
    'lin_regression': LinearRegression(),
    '2-mlp': MLPRegressor(
        hidden_layer_sizes=(64, 32), 
        max_iter=1000, 
        random_state=42
    )
}

## Beta-VAE baseline

In [27]:
X_train, y_train, X_test, y_test = prepare_data(
    ds=ds,
    embed_type="orig",
    factor="digit",
)

for model_name, clf_model in clf_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_classifier(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        model=clf_model
    )
    print("-----")

Model name: log_regression
--- Classification Results ---
Test F1: 0.9470
Test Accuracy: 0.9471

Bootstrap F1 Mean: 0.9470
Bootstrap F1 Median: 0.9471
   95% CI: [0.9426, 0.9514]
   68% CI: [0.9447, 0.9493]
   Err 95: [0.0044]

Bootstrap Accuracy Mean: 0.9471
Bootstrap Accuracy Median: 0.9472
   95% CI: [0.9427, 0.9515]
   68% CI: [0.9448, 0.9494]
   Err 95: [0.0044]
----------------------------------------
-----
Model name: 2-mlp
--- Classification Results ---
Test F1: 0.9772
Test Accuracy: 0.9772

Bootstrap F1 Mean: 0.9772
Bootstrap F1 Median: 0.9773
   95% CI: [0.9742, 0.9802]
   68% CI: [0.9756, 0.9787]
   Err 95: [0.0030]

Bootstrap Accuracy Mean: 0.9772
Bootstrap Accuracy Median: 0.9773
   95% CI: [0.9742, 0.9802]
   68% CI: [0.9756, 0.9787]
   Err 95: [0.0030]
----------------------------------------
-----


In [28]:
_, y_train, _, y_test = prepare_data(
    ds=ds,
    embed_type="orig",
    factor="b",
)
for model_name, reg_model in reg_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_regressor(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        model=reg_model
    )

Model name: lin_regression
--- Regression Results ---
Test R2: 0.9595
Bootstrap R2 Mean: 0.9595
Bootstrap R2 Median: 0.9595
   95% CI: [0.9580, 0.9609]
   68% CI: [0.9587, 0.9602]
   Err 95: [0.0015]
----------------------------------------
Model name: 2-mlp
--- Regression Results ---
Test R2: 0.9890
Bootstrap R2 Mean: 0.9890
Bootstrap R2 Median: 0.9890
   95% CI: [0.9885, 0.9895]
   68% CI: [0.9887, 0.9892]
   Err 95: [0.0005]
----------------------------------------


## Conditional Representations

In [29]:
X_train, y_train, X_test, y_test = prepare_data(
    ds=ds,
    embed_type="cond",
    factor="digit",
)

for model_name, clf_model in clf_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_classifier(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        model=clf_model
    )

Model name: log_regression
--- Classification Results ---
Test F1: 0.7475
Test Accuracy: 0.7487

Bootstrap F1 Mean: 0.7473
Bootstrap F1 Median: 0.7473
   95% CI: [0.7388, 0.7555]
   68% CI: [0.7429, 0.7515]
   Err 95: [0.0084]

Bootstrap Accuracy Mean: 0.7485
Bootstrap Accuracy Median: 0.7486
   95% CI: [0.7399, 0.7565]
   68% CI: [0.7443, 0.7528]
   Err 95: [0.0083]
----------------------------------------
Model name: 2-mlp
--- Classification Results ---
Test F1: 0.7887
Test Accuracy: 0.7888

Bootstrap F1 Mean: 0.7886
Bootstrap F1 Median: 0.7887
   95% CI: [0.7806, 0.7971]
   68% CI: [0.7842, 0.7929]
   Err 95: [0.0082]

Bootstrap Accuracy Mean: 0.7886
Bootstrap Accuracy Median: 0.7888
   95% CI: [0.7808, 0.7970]
   68% CI: [0.7843, 0.7930]
   Err 95: [0.0081]
----------------------------------------


In [30]:
_, y_train, _, y_test = prepare_data(
    ds=ds,
    embed_type="cond",
    factor="b",
)
for model_name, reg_model in reg_models.items():
    evaluate_embedding_regressor(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        model=reg_model
    )

--- Regression Results ---
Test R2: 0.9225
Bootstrap R2 Mean: 0.9225
Bootstrap R2 Median: 0.9225
   95% CI: [0.9198, 0.9250]
   68% CI: [0.9212, 0.9237]
   Err 95: [0.0026]
----------------------------------------
--- Regression Results ---
Test R2: 0.9583
Bootstrap R2 Mean: 0.9583
Bootstrap R2 Median: 0.9584
   95% CI: [0.9566, 0.9600]
   68% CI: [0.9575, 0.9592]
   Err 95: [0.0017]
----------------------------------------


# Conditional UMAP

In [31]:
import umap

In [32]:
X_train, y_train, X_test, y_test = prepare_data(
    ds=ds,
    embed_type="orig",
    factor="digit"
)

'''
print(f"\n=== Conditional UMAP Analysis. Guided by digit ===")
reducer = umap.UMAP(
    n_components=64, 
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
    transform_seed=42,
    n_jobs=1
)

reducer.fit(X_train, y=y_train)
X_train_emb = reducer.transform(X_train)
X_test_emb = reducer.transform(X_test)

np.savez_compressed("umap_embeddings.npz", X_train_emb=X_train_emb, X_test_emb=X_test_emb)
print("File saved successfully as 'umap_embeddings.npz'")
'''
data = np.load("./umap_embeddings.npz")

# Extract the training and testing embeddings
X_train_emb = data['X_train_emb']
X_test_emb = data['X_test_emb']

print(f"Loaded X_train_emb with shape: {X_train_emb.shape}")
print(f"Loaded X_test_emb with shape: {X_test_emb.shape}")

Loaded X_train_emb with shape: (48000, 64)
Loaded X_test_emb with shape: (10000, 64)


In [33]:
for model_name, clf_model in clf_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_classifier(
        X_train=X_train_emb,
        X_test=X_test_emb,
        y_train=y_train,
        y_test=y_test,
        model=clf_model
    )

Model name: log_regression
--- Classification Results ---
Test F1: 0.9634
Test Accuracy: 0.9634

Bootstrap F1 Mean: 0.9635
Bootstrap F1 Median: 0.9636
   95% CI: [0.9598, 0.9672]
   68% CI: [0.9616, 0.9654]
   Err 95: [0.0037]

Bootstrap Accuracy Mean: 0.9635
Bootstrap Accuracy Median: 0.9636
   95% CI: [0.9598, 0.9672]
   68% CI: [0.9616, 0.9654]
   Err 95: [0.0037]
----------------------------------------
Model name: 2-mlp
--- Classification Results ---
Test F1: 0.9632
Test Accuracy: 0.9632

Bootstrap F1 Mean: 0.9633
Bootstrap F1 Median: 0.9634
   95% CI: [0.9596, 0.9670]
   68% CI: [0.9614, 0.9652]
   Err 95: [0.0037]

Bootstrap Accuracy Mean: 0.9633
Bootstrap Accuracy Median: 0.9634
   95% CI: [0.9595, 0.9669]
   68% CI: [0.9614, 0.9652]
   Err 95: [0.0037]
----------------------------------------


In [34]:
_, y_train, _, y_test = prepare_data(
    ds=ds,
    embed_type="orig",
    factor="b"
)

for model_name, reg_model in reg_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_regressor(
        X_train=X_train_emb,
        X_test=X_test_emb,
        y_train=y_train,
        y_test=y_test,
        model=reg_model
    )

Model name: lin_regression
--- Regression Results ---
Test R2: 0.1637
Bootstrap R2 Mean: 0.1635
Bootstrap R2 Median: 0.1635
   95% CI: [0.1494, 0.1774]
   68% CI: [0.1563, 0.1703]
   Err 95: [0.0140]
----------------------------------------
Model name: 2-mlp
--- Regression Results ---
Test R2: 0.4122
Bootstrap R2 Mean: 0.4125
Bootstrap R2 Median: 0.4131
   95% CI: [0.3922, 0.4318]
   68% CI: [0.4025, 0.4223]
   Err 95: [0.0198]
----------------------------------------


## Residualisation (Linear)

In [35]:
X_train, y_train, X_test, y_test = prepare_data(
    ds=ds,
    embed_type="orig",
    factor="digit"
)

linear_model = LinearRegression()
X_train_res, X_test_res = resiualize_categorical(
    X_train, 
    y_train.reshape(-1, 1), 
    X_test, 
    y_test.reshape(-1, 1), 
    linear_model
)

In [36]:
for model_name, clf_model in clf_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_classifier(
        X_train=X_train_res,
        X_test=X_test_res,
        y_train=y_train,
        y_test=y_test,
        model=clf_model
    )

Model name: log_regression
--- Classification Results ---
Test F1: 0.0231
Test Accuracy: 0.1135

Bootstrap F1 Mean: 0.0232
Bootstrap F1 Median: 0.0231
   95% CI: [0.0208, 0.0257]
   68% CI: [0.0220, 0.0244]
   Err 95: [0.0025]

Bootstrap Accuracy Mean: 0.1135
Bootstrap Accuracy Median: 0.1135
   95% CI: [0.1072, 0.1200]
   68% CI: [0.1104, 0.1167]
   Err 95: [0.0064]
----------------------------------------
Model name: 2-mlp
--- Classification Results ---
Test F1: 0.8871
Test Accuracy: 0.8873

Bootstrap F1 Mean: 0.8871
Bootstrap F1 Median: 0.8871
   95% CI: [0.8808, 0.8930]
   68% CI: [0.8840, 0.8902]
   Err 95: [0.0061]

Bootstrap Accuracy Mean: 0.8873
Bootstrap Accuracy Median: 0.8873
   95% CI: [0.8809, 0.8932]
   68% CI: [0.8842, 0.8904]
   Err 95: [0.0061]
----------------------------------------


In [37]:
_, y_train, _, y_test = prepare_data(
    ds=ds,
    embed_type="orig",
    factor="b"
)

for model_name, reg_model in reg_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_regressor(
        X_train=X_train_res,
        X_test=X_test_res,
        y_train=y_train,
        y_test=y_test,
        model=reg_model
    )

Model name: lin_regression
--- Regression Results ---
Test R2: 0.9603
Bootstrap R2 Mean: 0.9603
Bootstrap R2 Median: 0.9603
   95% CI: [0.9588, 0.9618]
   68% CI: [0.9595, 0.9610]
   Err 95: [0.0015]
----------------------------------------
Model name: 2-mlp
--- Regression Results ---
Test R2: 0.9872
Bootstrap R2 Mean: 0.9872
Bootstrap R2 Median: 0.9872
   95% CI: [0.9866, 0.9877]
   68% CI: [0.9869, 0.9874]
   Err 95: [0.0005]
----------------------------------------


## Residualization (MLP Non-linear)

In [38]:
X_train, y_train, X_test, y_test = prepare_data(
    ds=ds,
    embed_type="orig",
    factor="digit"
)

mlp_model = MLPRegressor(
    hidden_layer_sizes=(256, 256), 
    max_iter=1000, 
    random_state=42
)

X_train_res, X_test_res = resiualize_categorical(
    X_train, 
    y_train.reshape(-1, 1), 
    X_test, 
    y_test.reshape(-1, 1), 
    mlp_model
)

In [39]:
for model_name, clf_model in clf_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_classifier(
        X_train=X_train_res,
        X_test=X_test_res,
        y_train=y_train,
        y_test=y_test,
        model=clf_model
    )

Model name: log_regression
--- Classification Results ---
Test F1: 0.9966
Test Accuracy: 0.9966

Bootstrap F1 Mean: 0.9966
Bootstrap F1 Median: 0.9966
   95% CI: [0.9955, 0.9977]
   68% CI: [0.9961, 0.9972]
   Err 95: [0.0011]

Bootstrap Accuracy Mean: 0.9966
Bootstrap Accuracy Median: 0.9966
   95% CI: [0.9955, 0.9977]
   68% CI: [0.9961, 0.9972]
   Err 95: [0.0011]
----------------------------------------
Model name: 2-mlp
--- Classification Results ---
Test F1: 0.9799
Test Accuracy: 0.9799

Bootstrap F1 Mean: 0.9799
Bootstrap F1 Median: 0.9799
   95% CI: [0.9771, 0.9826]
   68% CI: [0.9786, 0.9813]
   Err 95: [0.0027]

Bootstrap Accuracy Mean: 0.9799
Bootstrap Accuracy Median: 0.9799
   95% CI: [0.9771, 0.9826]
   68% CI: [0.9786, 0.9813]
   Err 95: [0.0028]
----------------------------------------


In [40]:
_, y_train, _, y_test = prepare_data(
    ds=ds,
    embed_type="orig",
    factor="b"
)

for model_name, reg_model in reg_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_regressor(
        X_train=X_train_res,
        X_test=X_test_res,
        y_train=y_train,
        y_test=y_test,
        model=reg_model
    )

Model name: lin_regression
--- Regression Results ---
Test R2: 0.9603
Bootstrap R2 Mean: 0.9603
Bootstrap R2 Median: 0.9603
   95% CI: [0.9589, 0.9618]
   68% CI: [0.9596, 0.9610]
   Err 95: [0.0014]
----------------------------------------
Model name: 2-mlp
--- Regression Results ---
Test R2: 0.9871
Bootstrap R2 Mean: 0.9871
Bootstrap R2 Median: 0.9870
   95% CI: [0.9865, 0.9876]
   68% CI: [0.9868, 0.9873]
   Err 95: [0.0005]
----------------------------------------


In [41]:
#RF Residualisation
from sklearn.ensemble import RandomForestRegressor

In [42]:
X_train, y_train, X_test, y_test = prepare_data(
    ds=ds,
    embed_type="orig",
    factor="digit"
)

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=RANDOM_STATE
)
X_train_res, X_test_res = resiualize_categorical(
    X_train, 
    y_train.reshape(-1, 1), 
    X_test, 
    y_test.reshape(-1, 1), 
    rf_model
)

In [43]:
for model_name, clf_model in clf_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_classifier(
        X_train=X_train_res,
        X_test=X_test_res,
        y_train=y_train,
        y_test=y_test,
        model=clf_model
    )

Model name: log_regression
--- Classification Results ---
Test F1: 0.0231
Test Accuracy: 0.1135

Bootstrap F1 Mean: 0.0232
Bootstrap F1 Median: 0.0231
   95% CI: [0.0208, 0.0257]
   68% CI: [0.0220, 0.0244]
   Err 95: [0.0025]

Bootstrap Accuracy Mean: 0.1135
Bootstrap Accuracy Median: 0.1135
   95% CI: [0.1072, 0.1200]
   68% CI: [0.1104, 0.1167]
   Err 95: [0.0064]
----------------------------------------
Model name: 2-mlp


--- Classification Results ---
Test F1: 0.8915
Test Accuracy: 0.8916

Bootstrap F1 Mean: 0.8916
Bootstrap F1 Median: 0.8916
   95% CI: [0.8857, 0.8976]
   68% CI: [0.8885, 0.8947]
   Err 95: [0.0059]

Bootstrap Accuracy Mean: 0.8917
Bootstrap Accuracy Median: 0.8917
   95% CI: [0.8859, 0.8975]
   68% CI: [0.8885, 0.8948]
   Err 95: [0.0058]
----------------------------------------


In [44]:
_, y_train, _, y_test = prepare_data(
    ds=ds,
    embed_type="orig",
    factor="b"
)

for model_name, reg_model in reg_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_regressor(
        X_train=X_train_res,
        X_test=X_test_res,
        y_train=y_train,
        y_test=y_test,
        model=reg_model
    )

Model name: lin_regression
--- Regression Results ---
Test R2: 0.9603
Bootstrap R2 Mean: 0.9603
Bootstrap R2 Median: 0.9603
   95% CI: [0.9588, 0.9617]
   68% CI: [0.9595, 0.9610]
   Err 95: [0.0015]
----------------------------------------
Model name: 2-mlp
--- Regression Results ---
Test R2: 0.9868
Bootstrap R2 Mean: 0.9868
Bootstrap R2 Median: 0.9868
   95% CI: [0.9863, 0.9874]
   68% CI: [0.9865, 0.9871]
   Err 95: [0.0005]
----------------------------------------
